In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 6 – Biological interpretation

*Owner: Part 4 (issue #4).* **What do the results of Stage 5 mean biologically?** Three interpretation methods, each linked back to the research question (*does PER3 genotype shift the time of day at which PER2, PER3 and NR1D1 peak?*):

1. **Enrichment (6.1):** what do the genes with a shared daily rhythm do, and which processes / blood cell types peak in the biological day vs the biological night?
2. **Network (6.2):** where do PER3, PER2 and NR1D1 sit in the clock's protein-interaction network (STRING), and does the **whole network** shift with genotype, not just the three primary genes?
3. **Literature (6.3):** do our numbers agree with published studies (all citations checked on PubMed)?

## Minimum evidence (lecturer's brief)
*Brief, stage 6: "Explain what the results mean biologically." Minimum evidence: "At least one interpretation method, linked back to the question."*

- [x] **At least one interpretation method**: GO/KEGG and cell-type enrichment (6.1), STRING/NetworkX network with a network-wide genotype test (6.2), literature comparison (6.3)
- [x] Each method **explicitly linked back to the research question** (see the *Link to the question* paragraph of each section)
- [x] Every citation verified (PubMed PMID and DOI, 2026-11-15); no invented references
- [x] Short interpretation of our own results
- [x] Report what the data does **and does not** show, with uncertainty (95 % CIs)

## Contract
**Inputs:** `results/tables/rhythmic_genes_SE.csv` (Stage 5 genome-wide scan, acrophase in **circadian phase**, 0 ≈ melatonin midpoint), `acrophase_genotype_tests.csv`, `acrophase_by_subject.csv` (Stage 5), `eda_subject_timing.csv` (Stage 4). Enrichment background: the 17,528 expressed genes (`data/processed/expressed_genes_SE.csv`, Stage 3). Expression for the network-wide test: `data.load_expression_matrix()` → `qc.expressed_genes()` (as in Stage 5).

**Implemented in `per3clock.interpret`:** `select_rhythmic_genes`, `load_gene_set_library`, `run_enrichment`, `fetch_string_network`, `build_network`, `network_centrality`, `expression_long`, `network_shift_test`, `plot_top_terms`, `plot_string_network`. Enrichr libraries and STRING responses are cached in `data/external/` (gitignored).

**Outputs:** tables `enrichment_rhythmic.csv`, `enrichment_rhythmic_by_phase.csv`, `enrichment_cell_types.csv`, `enrichment_sensitivity.csv`, `string_edges.csv`, `network_centrality.csv`, `network_genotype_shifts.csv`, `network_shift_test.csv`, `literature_comparison.csv`; figures `enrichment_top_terms.png`, `enrichment_cell_types.png`, `string_network.png`, `network_genotype_shifts.png`.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from per3clock import circstats, config, data, interpret, plotting, qc, rhythm

plotting.set_style()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

## 6.1 Enrichment of rhythmic genes

**Question:** what do the genes with a daily rhythm shared across people *do*? If PER3 genotype moved the clock, these are the processes whose timing would move with it.

**Method: over-representation analysis** (one-sided hypergeometric test, `interpret.run_enrichment`) against the **17,528 genes expressed on the array** (not the whole genome, which would inflate significance for anything blood-specific). Gene sets: Enrichr libraries GO Biological Process 2023 and KEGG 2021 Human (Xie et al. 2021), downloaded once with `gseapy` (Fang et al. 2023) and cached; terms with 5–500 background genes; Benjamini–Hochberg FDR within each library.

**Gene lists (fixed before looking at any enrichment result):** rhythmic at q < 0.05 **and** population amplitude ≥ 0.2 log2 (peak-to-trough ≥ ~32 %), because 55 % of expressed genes pass q < 0.05 alone (Stage 5) and such a list is too unspecific. The acrophases are strongly bimodal (Stage 5), so we also test the two clusters separately: **biological day** (circadian phase 6–12 h) and **biological evening/night** (phase 18–24 h, i.e. the hours before the melatonin midpoint). The amplitude threshold is varied in a sensitivity check below.

In [ ]:
scan = pd.read_csv(config.TABLES_DIR / "rhythmic_genes_SE.csv", index_col="gene")
background = pd.read_csv(config.PROCESSED_DIR / "expressed_genes_SE.csv")["gene"].tolist()
assert set(scan.index) == set(background), "scan and background should be the same genes"

Q_MAX, MIN_AMP = 0.05, 0.2      # fixed before looking at enrichment results
DAY, NIGHT = (6, 12), (18, 24)  # acrophase clusters, circadian phase (h)
gene_lists = {
    "All rhythmic (q<0.05, amplitude≥0.2)":
        interpret.select_rhythmic_genes(scan, q_max=Q_MAX, min_amplitude=MIN_AMP),
    "Peak in biological day (phase 6–12 h)":
        interpret.select_rhythmic_genes(scan, q_max=Q_MAX, min_amplitude=MIN_AMP, phase_window=DAY),
    "Peak in biological evening/night (phase 18–24 h)":
        interpret.select_rhythmic_genes(scan, q_max=Q_MAX, min_amplitude=MIN_AMP, phase_window=NIGHT),
}
fg = scan.loc[gene_lists["All rhythmic (q<0.05, amplitude≥0.2)"]]
phase_bins = pd.cut(fg["acrophase_h"] % 24, [0, 6, 12, 18, 24], right=False).value_counts().sort_index()
print(f"background: {len(background):,} expressed genes")
display(pd.Series({k: len(v) for k, v in gene_lists.items()}, name="n genes").to_frame())
print("Acrophase (circadian phase) of the rhythmic genes, 6-h bins:")
display(phase_bins.rename("n genes").to_frame())

In [ ]:
LIBRARIES = interpret.DEFAULT_LIBRARIES  # GO_Biological_Process_2023, KEGG_2021_Human
enr = {name: interpret.run_enrichment(genes, LIBRARIES, background)
       for name, genes in gene_lists.items()}

names = list(gene_lists)
enr[names[0]].to_csv(config.TABLES_DIR / "enrichment_rhythmic.csv", index=False)
pd.concat([enr[k].assign(gene_list=k) for k in names[1:]]).to_csv(
    config.TABLES_DIR / "enrichment_rhythmic_by_phase.csv", index=False)

show = ["gene_set", "term", "overlap", "fold_enrichment", "q"]
for name, t in enr.items():
    print(f"\n{name}: {len(gene_lists[name])} genes, {(t.q < 0.05).sum()} terms at q < 0.05")
    display(t.head(8)[show].round({"fold_enrichment": 2}).assign(q=lambda d: d.q.map("{:.1e}".format)))

In [ ]:
fig = interpret.plot_top_terms({k: enr[k] for k in names}, n=10)
plotting.save_fig(fig, "enrichment_top_terms")

**Which blood cells?** Stage 5 noted that the top rhythmic genes are lymphocyte markers, so the rhythm in whole blood may partly reflect **which cells are circulating** at each time of day rather than clocks ticking inside each cell. To test this directly we repeat the enrichment with cell-type marker sets: **Azimuth Cell Types 2021** (Enrichr library built from the Azimuth single-cell references, Hao et al. 2021), same background, terms up to 2,000 genes.

In [ ]:
cell_lib = {"Azimuth_Cell_Types_2021": interpret.load_gene_set_library("Azimuth_Cell_Types_2021")}
cell = {name: interpret.run_enrichment(genes, cell_lib, background, max_size=2000)
        for name, genes in gene_lists.items()}
pd.concat([t.assign(gene_list=k) for k, t in cell.items()]).to_csv(
    config.TABLES_DIR / "enrichment_cell_types.csv", index=False)
for name, t in cell.items():
    print(f"\n{name}: {(t.q < 0.05).sum()} cell types at q < 0.05")
    display(t.head(6)[["term", "overlap", "fold_enrichment", "q"]]
            .round({"fold_enrichment": 1}).assign(q=lambda d: d.q.map("{:.1e}".format)))
fig = interpret.plot_top_terms({k: cell[k] for k in names[1:]}, n=8)
plotting.save_fig(fig, "enrichment_cell_types")

**Where are the clock genes, and is the result robust?** The *KEGG Circadian rhythm* pathway has only ~30 genes, so it can be swamped by large immune/translation signals in a list of 2,000 genes. We look it up directly, and repeat the main enrichment with looser amplitude thresholds (0.1 and 0, i.e. q < 0.05 only).

In [ ]:
def circadian_rows(t):
    return t[t["term"].str.contains("ircadian") & (t["gene_set"] == "KEGG_2021_Human")]

display(circadian_rows(enr[names[0]])[["term", "overlap", "fold_enrichment", "p", "q", "genes"]])

sens = []
for amp in (0.2, 0.1, 0.0):
    genes = interpret.select_rhythmic_genes(scan, q_max=Q_MAX, min_amplitude=amp)
    t = interpret.run_enrichment(genes, LIBRARIES, background)
    circ = circadian_rows(t)
    sens.append({
        "min_amplitude": amp, "n_genes": len(genes), "n_terms_q05": int((t.q < 0.05).sum()),
        "top_3_terms": " | ".join(t["term"].head(3).str.replace(r"\s*\(GO:\d+\)", "", regex=True)),
        "kegg_circadian_overlap": circ["overlap"].iloc[0] if len(circ) else "",
        "kegg_circadian_q": circ["q"].iloc[0] if len(circ) else np.nan,
    })
sens = pd.DataFrame(sens)
sens.to_csv(config.TABLES_DIR / "enrichment_sensitivity.csv", index=False)
display(sens)

**Enrichment results.**
- **Two daily programmes.** 1,929 of the 2,007 strongly rhythmic genes (96 %) peak in one of two windows: the **biological day** (phase 6–12 h, 1,149 genes) or the **biological evening/night** (phase 18–24 h, 780 genes). Very few peak in between.
- **Evening/night genes are mainly T-cell and ribosome genes.** The top terms are ribosome and cytoplasmic translation (KEGG Ribosome 50/131 genes, 8.6-fold, q = 2 × 10⁻³¹). The KEGG *Coronavirus disease* term is the same signal: 45 of its 48 overlapping genes are ribosomal proteins. The cell-type test is clearer: **naive CD4⁺ T cells** (11/18 markers, 14-fold, q = 4 × 10⁻⁹), naive CD8⁺ and regulatory T cells. Naive T cells carry many ribosomal transcripts, so the two results probably describe the same cells.
- **Day genes are mainly innate and cytotoxic immune genes:** complement and coagulation (19/67, q = 1 × 10⁻⁵), NK-cell-mediated cytotoxicity (KIR genes, GZMB, PRF1; q = 0.04) and haematopoietic lineage. The cell-type test points to **CD56-dim NK cells** (7/9 markers, 12-fold, q = 3 × 10⁻⁵) and monocyte/macrophage markers.
- **The core clock does not stand out as a pathway.** KEGG *Circadian rhythm* is not enriched (2/28 genes in the main list; 16/28 with q < 0.05 alone, q = 0.75). Its genes are individually rhythmic (Stage 5), but there are only ~30 of them and their amplitudes are small, so in whole blood they are swamped by much larger cell-composition rhythms.
- **Robust to the threshold.** Ribosome/translation stays the top signal with amplitude ≥ 0.1 (7,037 genes) and with q < 0.05 alone (9,577 genes) (`enrichment_sensitivity.csv`).

**Link to the question.** Most of the daily rhythm in whole blood reflects **which white cells are circulating** at each hour (naive T cells at night, NK cells and monocytes by day), not only clocks ticking inside each cell. PER2, PER3 and NR1D1 peak timings measured in whole blood are therefore a mixture of cellular clock timing and cell-trafficking timing. A PER3-genotype effect on either would appear as a shift. The reverse also holds: a real but small intracellular shift could be diluted by the much larger cell-composition rhythm.

## 6.2 STRING network of clock genes

**Question:** how are PER3, PER2 and NR1D1 wired into the clock, and does the clock **as a whole** run earlier in PER3 5/5 carriers?

**Method.** Seed genes: the 8 clock genes of the project plus well-known clock-controlled transcription factors and clock kinases (Takahashi 2017): CRY2, NPAS2, DBP, TEF, HLF, NFIL3, BHLHE40/41 (DEC1/2), RORA, RORC, CIART, CSNK1D/E, keeping those expressed on the array. Interactions from the STRING API (v12.5; Szklarczyk et al. 2023), combined score ≥ 0.4 (medium confidence), human. Our gene symbols are mapped to STRING identifiers first, so renamed genes (ARNTL = BMAL1) keep our names. NetworkX: degree, strength (sum of scores), betweenness, closeness and Louvain communities (fixed seed). Nodes are coloured by their **acrophase in our data** and sized by amplitude.

In [ ]:
seed_genes = [g for g in interpret.CLOCK_NETWORK_GENES if g in set(background)]
print(f"{len(seed_genes)} seed genes expressed on the array; not expressed:",
      sorted(set(interpret.CLOCK_NETWORK_GENES) - set(seed_genes)))

edges = interpret.fetch_string_network(seed_genes, required_score=400)
edges.to_csv(config.TABLES_DIR / "string_edges.csv", index=False)
G = interpret.build_network(edges)
cent = interpret.network_centrality(G).join(scan[["amplitude", "acrophase_h", "q"]])
cent.to_csv(config.TABLES_DIR / "network_centrality.csv")

n = G.number_of_nodes()
print(f"{n} nodes, {G.number_of_edges()} edges "
      f"(density {2 * G.number_of_edges() / (n * (n - 1)):.2f}); "
      f"{(edges.score >= 0.7).sum()} edges with score ≥ 0.7, {(edges.score >= 0.9).sum()} with ≥ 0.9")
display(cent.round(3))

In [ ]:
fig = interpret.plot_string_network(
    G, scan, title="STRING network of clock genes (score ≥ 0.4); colour = acrophase in our data")
plotting.save_fig(fig, "string_network")

In [ ]:
# Which evidence supports the links between the three primary genes?
prim = set(config.GENES_PRIMARY)
display(edges[edges.source.isin(prim) & edges.target.isin(prim)].round(3))

# Peak order of the rhythmic network genes (circadian phase, h; 0 ≈ melatonin midpoint)
order = cent[cent.q < 0.05].assign(phase=lambda d: (d.acrophase_h + 6) % 24 - 6)
display(order.sort_values("phase")[["phase", "amplitude", "q", "community"]].round(3).T)

### 6.2b Does the whole clock network shift with PER3 genotype?

Stage 5 tested PER2, PER3 and NR1D1 one at a time. The network lets us ask a sharper question: if PER3 genotype moved the clock, **all rhythmic clock-network genes** should shift in the same direction. We fit per-subject cosinors (`rhythm.fit_subject_cosinors`) to every network gene rhythmic at q < 0.05 in the genome-wide scan and compare genotypes with the Stage 5 methods (`circstats.genotype_shift_table`: circular mean difference 5/5 − 4/4, 95 % bootstrap CI, permutation test, BH across genes). The **network-level** test (`interpret.network_shift_test`) averages the shifts over genes and permutes genotype labels **between subjects**, so the strong correlation between clock genes within a person is kept and 14 genes are not counted as 14 independent pieces of evidence. Same sample sets as Stage 5 (clock time: 24 subjects; circadian phase: 23 subjects, AF0182 excluded).

In [ ]:
expr = data.load_expression_matrix()
annot = data.load_probe_annotation()
expressed, _ = qc.expressed_genes(expr, annot)
net_genes = [g for g in seed_genes if scan.loc[g, "q"] < 0.05]
print(len(net_genes), "rhythmic network genes:", ", ".join(net_genes))

shift_tabs, net_tests = [], []
for time_base, samples, time_col in [
    ("clock_time", data.analysis_samples(), "clock_time_h"),
    ("circadian_phase", data.analysis_samples(for_phase_analysis=True), "circadian_phase_h"),
]:
    long = interpret.expression_long(expressed, samples, net_genes)
    fits = rhythm.fit_subject_cosinors(long, genes=net_genes, time_col=time_col)
    shift_tabs.append(circstats.genotype_shift_table(fits, by=("gene",)).assign(time_base=time_base))
    res = interpret.network_shift_test(fits)
    net_tests.append({"time_base": time_base, **{k: v for k, v in res.items() if k != "per_gene"}})

net_shifts = pd.concat(shift_tabs, ignore_index=True)
net_shifts.to_csv(config.TABLES_DIR / "network_genotype_shifts.csv", index=False)
net_test = pd.DataFrame(net_tests)
net_test.to_csv(config.TABLES_DIR / "network_shift_test.csv", index=False)

display(net_shifts.pivot(index="gene", columns="time_base",
                         values=["shift_h", "ci_lower_h", "ci_upper_h", "permutation_q"])
        .reindex(net_genes).round(2))
display(net_test.round(3))

In [ ]:
fig = plotting.plot_shift_forest(
    net_shifts.sort_values(["time_base", "gene"], key=lambda s: s.map(
        {g: i for i, g in enumerate(net_genes)}) if s.name == "gene" else s),
    label_cols=("gene", "time_base"))
fig.axes[0].get_legend().remove()  # all rows are primary analyses here
fig.axes[0].set_title("Clock-network genes: genotype shift in peak time (5/5 − 4/4), 95 % CI",
                      loc="left")
plotting.save_fig(fig, "network_genotype_shifts")

**Network results.**
- **The clock is one dense module, and PER3 is in its core.** The 20 seed genes share 171 of 190 possible STRING links (density 0.90; 115 links with score ≥ 0.9). PER3 links to **all 19** other genes, tied for the highest degree with PER2, NR1D1, CRY1/2, CLOCK, ARNTL and NPAS2. Betweenness is near zero for every gene. In such a dense, redundant network, centrality cannot separate "important" from "peripheral" clock genes. What it shows is that PER3 is not an isolated gene.
- **How the three primary genes are linked.** PER2–PER3 is supported by strong experimental evidence (score 0.998; experimental 0.97), consistent with PER proteins acting together in the clock's repressor complex (Takahashi 2017). NR1D1–PER2 and NR1D1–PER3 are supported mainly by text mining (experimental 0.65 and 0.27). These are regulatory links in the same feedback loops rather than direct physical binding.
- **The peak order in our data matches the clock's design.** Relative to the melatonin midpoint, NR1D1 peaks at −1.6 h, NR1D2 +0.3 h, PER2 +1.3 h, PER3 +2.4 h and PER1 +3.3 h. DBP, NFIL3 and BHLHE40 peak in the biological day (+8.5 to +10.7 h). The clock-network genes are rhythmic in blood in a coherent, staggered order.
- **The whole network does not shift with genotype on the internal-clock axis.** Fourteen network genes are rhythmic (q < 0.05).
  - *Clock time:* 12 of 14 peak earlier in 5/5, mean shift **−1.0 h (95 % CI −3.5 to +0.6; permutation p = 0.28)**. No single gene has q < 0.10.
  - *Circadian phase (relative to melatonin):* 7 of 14 earlier, mean shift **−0.04 h (95 % CI −1.8 to +1.1; p = 0.95)**.
  - Taken alone, "12 of 14 genes earlier" looks convincing. It is not, because clock genes within one person move together. The subject-level permutation accounts for this.

**Link to the question.** On clock time the whole network tends to run earlier in 5/5 carriers, by about the same amount as their earlier wake time (1.2 h) and melatonin timing (1.3 h; Stage 4). Measured against each person's own melatonin rhythm, the network lines up between genotypes. **If PER3 5/5 has any effect, it is consistent with the whole clock (and sleep schedule) being set earlier, not with PER2/PER3/NR1D1 being re-timed relative to the internal clock.** The clock-time effect is not statistically supported. Its CI includes zero and also includes shifts of up to ~3.5 h earlier.

## 6.3 Literature comparison

We compare our estimates with published studies. All references were checked on PubMed (PMID and DOI) on 2026-11-15 and are in `docs/references.bib`; the published values are quoted from the abstracts. Our values are computed here from the Stage 4–6 tables.

In [ ]:
acro = pd.read_csv(config.TABLES_DIR / "acrophase_by_subject.csv")
acro = acro[acro.sample_set == "Primary"]

def cmean(x):
    return circstats.circular_mean_h(x.to_numpy())

def fmt_clock(h):
    h = h % 24
    return f"{int(h):02d}:{int(round((h % 1) * 60)) % 60:02d}"

peak = acro.groupby(["gene", "time_base"])["acrophase_h"].apply(cmean)
tests = pd.read_csv(config.TABLES_DIR / "acrophase_genotype_tests.csv").set_index(["gene", "time_base"])
timing = pd.read_csv(config.TABLES_DIR / "eda_subject_timing.csv")
timing = timing[(timing.sleep_condition == config.MAIN_CONDITION)
                & timing.subject.isin(data.analysis_samples().subject)]
wake = timing.groupby("per3_genotype")["wake_clock_h"].apply(cmean)
mel = timing.groupby("per3_genotype")["phase0_clock_h"].apply(cmean)
wake_diff = circstats.circular_diff_h(wake["5/5"], wake["4/4"])
mel_diff = circstats.circular_diff_h(mel["5/5"], mel["4/4"])

def shift(gene, tb):
    r = tests.loc[(gene, tb)]
    return f"{r.shift_h:+.1f} h [{r.ci_lower_h:+.1f}, {r.ci_upper_h:+.1f}]"

ct = net_test.set_index("time_base")
def top_cells(t, n=3):  # cell-type names without ontology ids / numbered subclusters
    names_ = t[t.q < 0.05]["term"].str.replace(r"( \d+)? CL\d+$", "", regex=True)
    return names_.drop_duplicates().head(n).tolist()

day_top, night_top = top_cells(cell[names[1]]), top_cells(cell[names[2]])

lit = pd.DataFrame([
    {"topic": "Number of circadian transcripts, sufficient sleep",
     "published": "1,855 genes with a circadian profile (after 1 wk sufficient sleep; same dataset)",
     "source": "Möller-Levet et al. 2013 (PMID 23440187)",
     "ours": f"{len(gene_lists[names[0]]):,} at q<0.05 & amplitude≥0.2 "
             f"({int((scan.q < 0.05).sum()):,} at q<0.05 alone)",
     "agreement": "Yes, with an amplitude filter (different method: population cosinor)"},
    {"topic": "PER3 peak time (clock time)",
     "published": "06:05 (leukocytes, constant routine, n = 24)",
     "source": "Archer et al. 2008 (PMID 18517031)",
     "ours": f"{fmt_clock(peak[('PER3', 'clock_time')])} (whole blood, n = 24)",
     "agreement": "Close (within one 3-h sampling interval)"},
    {"topic": "PER2 peak time (clock time)",
     "published": "08:06",
     "source": "Archer et al. 2008 (PMID 18517031)",
     "ours": f"{fmt_clock(peak[('PER2', 'clock_time')])}",
     "agreement": "Ours earlier; different protocol and tissue (see text)"},
    {"topic": "PER3 relative to melatonin",
     "published": "PER3 peaks ~1.8 h after the melatonin peak (06:05 vs 04:20)",
     "source": "Archer et al. 2008 (PMID 18517031)",
     "ours": f"PER3 at circadian phase {peak[('PER3', 'circadian_phase')]:+.1f} h "
             "(0 ≈ melatonin midpoint)",
     "agreement": "Yes"},
    {"topic": "PER3 genotype and timing of PER3 mRNA / melatonin",
     "published": "Circadian rhythms of melatonin, cortisol and peripheral PER3 mRNA not affected "
                  "by PER3 genotype (n = 24, selected by genotype)",
     "source": "Viola et al. 2007 (PMID 17346965)",
     "ours": f"PER3 shift {shift('PER3', 'circadian_phase')} (phase); "
             f"{shift('PER3', 'clock_time')} (clock time); no gene significant",
     "agreement": "Yes (both null, both small samples)"},
    {"topic": "PER3 5/5 and sleep–wake timing",
     "published": "5/5 associated with morning preference and earlier wake time and bedtime (n = 675)",
     "source": "Lázár et al. 2012 (PMID 22324552); Archer et al. 2003 (PMID 12841365)",
     "ours": f"5/5 woke {abs(wake_diff):.1f} h {'earlier' if wake_diff < 0 else 'later'}, "
             f"melatonin midpoint {abs(mel_diff):.1f} h {'earlier' if mel_diff < 0 else 'later'} (Stage 4)",
     "agreement": "Yes in direction (not tested here)"},
    {"topic": "Whole clock network vs genotype",
     "published": "(no published network-level test in this dataset)",
     "source": "this study",
     "ours": f"clock time {ct.loc['clock_time', 'mean_shift_h']:+.1f} h "
             f"[{ct.loc['clock_time', 'ci_lower_h']:+.1f}, {ct.loc['clock_time', 'ci_upper_h']:+.1f}], "
             f"p = {ct.loc['clock_time', 'p_mean_shift']:.2f}; phase "
             f"{ct.loc['circadian_phase', 'mean_shift_h']:+.2f} h, p = {ct.loc['circadian_phase', 'p_mean_shift']:.2f}",
     "agreement": "n/a"},
    {"topic": "Naive T cells: high at night, low during the day",
     "published": "Naive CD4+/CD8+ T cells have a daytime nadir, driven by morning cortisol via "
                  "CXCR4; CD4 naive cells have the most robust rhythm",
     "source": "Dimitrov et al. 2009 (PMID 19293427); Besedovsky et al. 2014 (PMID 24051033); "
               "Ackermann et al. 2012 (PMID 22754039)",
     "ours": "Genes peaking in the biological evening/night are enriched for " + ", ".join(night_top),
     "agreement": "Yes"},
    {"topic": "Cytotoxic effector cells peak during the day",
     "published": "Effector CD8+ T cells peak during daytime (epinephrine)",
     "source": "Dimitrov et al. 2009 (PMID 19293427)",
     "ours": "Genes peaking in the biological day are enriched for " + ", ".join(day_top),
     "agreement": "Consistent (cytotoxic NK markers overlap with effector CD8 markers)"},
])
lit.to_csv(config.TABLES_DIR / "literature_comparison.csv", index=False)
display(lit)

**Literature comparison.**
- **Scale of rhythmicity.** Möller-Levet et al. (2013) reported 1,855 circadian transcripts in this dataset after sufficient sleep. Our amplitude-filtered list (2,007) is of the same size, while q < 0.05 alone (9,577) is much larger. The authors' method was more conservative than our population cosinor, which detects very small but consistent rhythms (Stage 5).
- **Peak times.**
  - Our PER3 peak (05:42) is within half an hour of the 06:05 reported in leukocytes by Archer et al. (2008), and PER3 peaks ~1.6 h after the melatonin midpoint in our data (mean of per-subject peaks; the population cosinor in 6.2 gives +2.4 h) versus ~1.8 h after the melatonin peak in theirs.
  - Our PER2 peak (05:13) is ~3 h earlier than theirs (08:06). Possible reasons: whole blood vs isolated leukocytes, extended wakefulness after a week of sleep extension vs a constant routine, 3-h sampling, and PER2's small amplitude. We cannot tell which.
- **Genotype.** Our null result agrees with Viola et al. (2007), who found that PER3 genotype did **not** affect the circadian timing of melatonin, cortisol or peripheral PER3 mRNA in 24 people selected by genotype. It also fits the larger cohort of Lázár et al. (2012, n = 675), where 5/5 carriers wake and go to bed earlier. That is the same direction as the earlier wake times of our 5/5 group, and it matches PER3's reported association with morning preference (Archer et al. 2003).
- **Immune rhythms.** The day/night split of our enrichment matches the physiology of circulating leukocytes. Naive T cells are high at night and fall in the morning as cortisol drives them out of the blood via CXCR4 (Dimitrov et al. 2009; Besedovsky et al. 2014). Naive CD4 cells have the most robust rhythm (Ackermann et al. 2012). Cytotoxic effector cells peak during the day (Dimitrov et al. 2009).

### References (verified on PubMed, 2026-11-15)
- Ackermann K, Revell VL, Lao O, et al. (2012) Diurnal rhythms in blood cell populations and the effect of acute sleep deprivation in healthy young men. *Sleep* 35(7):933–940. doi:10.5665/sleep.1954 (PMID 22754039)
- Archer SN, Robilliard DL, Skene DJ, et al. (2003) A length polymorphism in the circadian clock gene Per3 is linked to delayed sleep phase syndrome and extreme diurnal preference. *Sleep* 26(4):413–415. doi:10.1093/sleep/26.4.413 (PMID 12841365)
- Archer SN, Viola AU, Kyriakopoulou V, von Schantz M, Dijk DJ (2008) Inter-individual differences in habitual sleep timing and entrained phase of endogenous circadian rhythms of BMAL1, PER2 and PER3 mRNA in human leukocytes. *Sleep* 31(5):608–617. doi:10.1093/sleep/31.5.608 (PMID 18517031)
- Besedovsky L, Born J, Lange T (2014) Endogenous glucocorticoid receptor signaling drives rhythmic changes in human T-cell subset numbers and the expression of the chemokine receptor CXCR4. *FASEB J* 28(1):67–75. doi:10.1096/fj.13-237958 (PMID 24051033)
- Dimitrov S, Benedict C, Heutling D, et al. (2009) Cortisol and epinephrine control opposing circadian rhythms in T cell subsets. *Blood* 113(21):5134–5143. doi:10.1182/blood-2008-11-190769 (PMID 19293427)
- Fang Z, Liu X, Peltz G (2023) GSEApy: a comprehensive package for performing gene set enrichment analysis in Python. *Bioinformatics* 39(1):btac757. doi:10.1093/bioinformatics/btac757 (PMID 36426870)
- Hao Y, Hao S, Andersen-Nissen E, et al. (2021) Integrated analysis of multimodal single-cell data. *Cell* 184(13):3573–3587. doi:10.1016/j.cell.2021.04.048 (PMID 34062119)
- Lázár AS, Slak A, Lo JC, et al. (2012) Sleep, diurnal preference, health, and psychological well-being: a prospective single-allelic-variation study. *Chronobiol Int* 29(2):131–146. doi:10.3109/07420528.2011.641193 (PMID 22324552)
- Möller-Levet CS, Archer SN, Bucca G, et al. (2013) Effects of insufficient sleep on circadian rhythmicity and expression amplitude of the human blood transcriptome. *PNAS* 110(12):E1132–E1141. doi:10.1073/pnas.1217154110 (PMID 23440187)
- Szklarczyk D, Kirsch R, Koutrouli M, et al. (2023) The STRING database in 2023. *Nucleic Acids Res* 51(D1):D638–D646. doi:10.1093/nar/gkac1000 (PMID 36370105)
- Takahashi JS (2017) Transcriptional architecture of the mammalian circadian clock. *Nat Rev Genet* 18(3):164–179. doi:10.1038/nrg.2016.150 (PMID 27990019)
- Viola AU, Archer SN, James LM, et al. (2007) PER3 polymorphism predicts sleep structure and waking performance. *Curr Biol* 17(7):613–618. doi:10.1016/j.cub.2007.01.073 (PMID 17346965)
- Xie Z, Bailey A, Kuleshov MV, et al. (2021) Gene set knowledge discovery with Enrichr. *Curr Protoc* 1(3):e90. doi:10.1002/cpz1.90 (PMID 33780170)

## Interpretation

**What the results mean biologically.** Human blood has a strong daily transcriptional programme, but much of it reflects white-cell traffic: naive T cells (with their ribosome-rich transcripts) circulate at night, and NK cells, monocytes and complement components dominate by day. This matches known cortisol- and adrenaline-driven immune rhythms. Within that, the molecular clock is clearly running. NR1D1, NR1D2, PER2, PER3 and PER1 peak in the expected staggered order, from just before to just after the melatonin midpoint, and PER3's timing agrees with published leukocyte data. In the STRING network PER3 sits in the dense core of the clock, directly linked to PER2 and connected to NR1D1 through the clock's feedback loops.

**Back to the research question.** If PER3 genotype re-timed the clock, the clock-network genes should shift together. On clock time they tend to: 12 of 14 genes peak earlier in 5/5 carriers, by about 1 h on average. But the network-level test is not significant (−1.0 h, 95 % CI −3.5 to +0.6, p = 0.28), and on the internal-clock axis the shift vanishes (−0.04 h, CI −1.8 to +1.1). This pattern, together with the earlier wake and melatonin times of the 5/5 group and the null result of Viola et al. (2007), suggests that **any PER3 effect is a shift of the whole sleep–clock system, not a change in when clock genes peak relative to the internal clock.**

**What the data do not show.** They do not show that PER3 genotype shifts clock-gene timing, and with 11–13 people per group they cannot rule out a shift of 1–3 h on clock time. Enrichment and STRING describe associations and prior knowledge, not mechanisms. The whole-blood signal mixes intracellular clocks with cell composition, which we could not separate without cell counts or sorted cells.